This script will read in a parquet file to be predicted on, a model to predict with, predict on the parquet file and thurn turn those predictions back to a tif in our 1km grid and epsg 3413. 

In [7]:
import pandas as pd
import numpy as np

# --------------------------
# Load parquet + clean names
# --------------------------
path = "/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined_test/predictors_2004_07.parquet"
df = pd.read_parquet(path)

def clean_col(c):
    return c.split("__", 1)[1] if "__" in c else c

df = df.rename(columns={c: clean_col(c) for c in df.columns})


# --------------------------
# Explicit rename mapping
# --------------------------
rename_map = {
    'LST_Day_mean'           : 'LST_day_mean',
    'LST_Night_mean'         : 'LST_night_mean',
    'N_0_100cm'              : 'N_N_0_100cm',
    'alpha_0_100cm'          : 'alpha_ALFA_0_100cm',
    'crit_wilt_0_100cm'      : 'crit_wilt_CRIT-WILT_0_100cm',
    'field_crit_0_100cm'     : 'field_crit_FIELD-CRIT_0_100cm',
    'Fpar'                   : 'fpar',
    'ksat_0_100cm'           : 'ksat_Ksat_0_100cm',
    'Lai'                    : 'lai',
    'ormc_0_100cm'           : 'ormc_ORMC_0_100cm',
    'satfield_0_100cm'       : 'satfield_SAT-FIELD_0_100cm',
    'band1'                  : 'sm_rootzone',
    'band2'                  : 'sm_surface',
    'wcavail_0_100cm'        : 'wcavail_WCavail_0_100cm',
    'wcpf2_0_100cm'          : 'wcpf2_WCpF2_0_100cm',
    'wcpf3_0_100cm'          : 'wcpf3_WCpF3_0_100cm',
    'wcpf4_2_0_100cm'        : 'wcpf4_2_WCpF4-2_0_100cm',
    'wcres_0_100cm'          : 'wcres_WCres_0_100cm',
    'wcsat_0_100cm'          : 'wcsat_WCsat_0_100cm'
}

# Apply renaming
df = df.rename(columns=rename_map)

#target columns
t = [
    'EVI', 'NDVI', 'sur_refl_b01',
    'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b07',
    'NDWI', 'pdsi', 'srad',
    'tmean_C', 'vap', 'vs',
    'swe', 'aet', 'pet',
    'def', 'bdod_0_100cm', 'cec_0_100cm',
    'cfvo_0_100cm', 'clay_0_100cm', 'nitrogen_0_100cm',
    'ocd_0_100cm', 'phh2o_0_100cm', 'sand_0_100cm',
    'silt_0_100cm', 'soc_0_100cm', 'co2_cont',
    'ALT', 'month', 'lai',
    'fpar', 'Percent_NonTree_Vegetation', 'Percent_NonVegetated',
    'Percent_Tree_Cover', 'sm_surface', 'sm_rootzone',
    'snow_cover', 'snow_depth', 'soil_temperature_level_1',
    'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4',
    'N_N_0_100cm', 'alpha_ALFA_0_100cm', 'crit_wilt_CRIT-WILT_0_100cm',
    'field_crit_FIELD-CRIT_0_100cm', 'ksat_Ksat_0_100cm', 'ormc_ORMC_0_100cm',
    'satfield_SAT-FIELD_0_100cm', 'wcavail_WCavail_0_100cm',
    'wcpf2_WCpF2_0_100cm', 'wcpf3_WCpF3_0_100cm', 'wcpf4_2_WCpF4-2_0_100cm',
    'wcres_WCres_0_100cm', 'wcsat_WCsat_0_100cm', 'month_sin',
    'month_cos', 'tpi', 'slope', 'elevation', 'aspect',
    'LST_day_mean', 'LST_night_mean'
]

#add sin and cos of month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

# --------------------------
# Find missing columns
# --------------------------
missing = sorted([col for col in t if col not in df.columns])

print("Missing columns ({}):".format(len(missing)))
for col in missing:
    print(col)


Missing columns (0):


In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import joblib
import rasterio as rio
from rasterio.transform import Affine

# ============================================================
# CONFIG: paths you MUST edit
# ============================================================

PARQUET_PATH = "/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined_test/predictors_2004_07.parquet"

# Path to the final NEE model saved by your LOSO script:
#   joblib.dump(final_model, os.path.join(save_path, 'nee_model_final.joblib'))
NEE_MODEL_PATH = "/explore/nobackup/people/spotter5/anna_v/v2/models/final_models_loso_20251201_174727/nee_model_final.joblib"

# If you saved your feature list in run_log.json, you can load it.
# For now, either:
#   1) set FINAL_FEATURES manually, or
#   2) load them from your JSON log.
RUN_LOG_JSON = "/explore/nobackup/people/spotter5/anna_v/v2/models/final_models_loso_20251201_174727/run_log.json"

# Output GeoTIFF path (NEE prediction)
OUT_TIF = "/explore/nobackup/people/spotter5/anna_v/v2/predictions_test/nee_pred_2004_07.tif"
os.makedirs("/explore/nobackup/people/spotter5/anna_v/v2/predictions_test", exist_ok = True)

# CRS for x/y coordinates. Given your pipeline, this is almost certainly EPSG:3413.
OUT_CRS = "EPSG:3413"

# ============================================================
# 1. Load parquet + clean column names
# ============================================================

df = pd.read_parquet(PARQUET_PATH)

def clean_col(c):
    return c.split("__", 1)[1] if "__" in c else c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

# ============================================================
# 2. Apply explicit rename mapping
# ============================================================

rename_map = {
    # 'LST_Day_mean'           : 'LST_day_mean',
    # 'LST_Night_mean'         : 'LST_night_mean',
    'N_0_100cm'              : 'N_N_0_100cm',
    'alpha_0_100cm'          : 'alpha_ALFA_0_100cm',
    'crit_wilt_0_100cm'      : 'crit_wilt_CRIT-WILT_0_100cm',
    'field_crit_0_100cm'     : 'field_crit_FIELD-CRIT_0_100cm',
    'Fpar'                   : 'fpar',
    'ksat_0_100cm'           : 'ksat_Ksat_0_100cm',
    'Lai'                    : 'lai',
    'ormc_0_100cm'           : 'ormc_ORMC_0_100cm',
    'satfield_0_100cm'       : 'satfield_SAT-FIELD_0_100cm',
    'band1'                  : 'sm_rootzone',
    'band2'                  : 'sm_surface',
    'wcavail_0_100cm'        : 'wcavail_WCavail_0_100cm',
    'wcpf2_0_100cm'          : 'wcpf2_WCpF2_0_100cm',
    'wcpf3_0_100cm'          : 'wcpf3_WCpF3_0_100cm',
    'wcpf4_2_0_100cm'        : 'wcpf4_2_WCpF4-2_0_100cm',
    'wcres_0_100cm'          : 'wcres_WCres_0_100cm',
    'wcsat_0_100cm'          : 'wcsat_WCsat_0_100cm'
}
df = df.rename(columns=rename_map)

# ============================================================
# 3. Add month_sin and month_cos
# ============================================================

if "month" not in df.columns:
    raise ValueError("Column 'month' not found in dataframe; cannot compute month_sin/month_cos.")

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)

# ============================================================
# 4. Define the feature list (must match training)
# ============================================================

# Option A: load from run_log.json (recommended)
import json
with open(RUN_LOG_JSON, "r") as f:
    log_data = json.load(f)
FINAL_FEATURES = log_data["features_used"]

# If you prefer to hardcode, comment out the above and do:
# FINAL_FEATURES = [
#     'EVI', 'NDVI', 'sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b07',
#     'NDWI', 'pdsi', 'srad', 'tmean_C', 'vap', 'vs', 'swe', 'aet', 'pet', 'def',
#     'bdod_0_100cm', 'cec_0_100cm', 'cfvo_0_100cm', 'clay_0_100cm', 'nitrogen_0_100cm',
#     'ocd_0_100cm', 'phh2o_0_100cm', 'sand_0_100cm', 'silt_0_100cm', 'soc_0_100cm',
#     'co2_cont', 'ALT', 'month', 'lai', 'fpar', 'Percent_NonTree_Vegetation',
#     'Percent_NonVegetated', 'Percent_Tree_Cover', 'sm_surface', 'sm_rootzone',
#     'snow_cover', 'snow_depth', 'soil_temperature_level_1', 'soil_temperature_level_2',
#     'soil_temperature_level_3', 'soil_temperature_level_4', 'N_N_0_100cm',
#     'alpha_ALFA_0_100cm', 'crit_wilt_CRIT-WILT_0_100cm',
#     'field_crit_FIELD-CRIT_0_100cm', 'ksat_Ksat_0_100cm', 'ormc_ORMC_0_100cm',
#     'satfield_SAT-FIELD_0_100cm', 'wcavail_WCavail_0_100cm',
#     'wcpf2_WCpF2_0_100cm', 'wcpf3_WCpF3_0_100cm', 'wcpf4_2_WCpF4-2_0_100cm',
#     'wcres_WCres_0_100cm', 'wcsat_WCsat_0_100cm', 'month_sin', 'month_cos',
#     'tpi', 'slope', 'elevation', 'aspect', 'LST_day_mean', 'LST_night_mean'
# ]

# Ensure all features exist
for col in FINAL_FEATURES:
    if col not in df.columns:
        print(f"Missing column {col} and filling with nan")
        df[col] = np.nan  # or raise, depending on how strict you want to be

X = df[FINAL_FEATURES].copy()

# Coerce to numeric
for c in X.columns:
    if not np.issubdtype(X[c].dtype, np.number):
        X[c] = pd.to_numeric(X[c], errors="coerce")

# ============================================================
# 5. Load NEE model and predict
# ============================================================

if not os.path.exists(NEE_MODEL_PATH):
    raise FileNotFoundError(f"NEE model file not found: {NEE_MODEL_PATH}")

nee_model = joblib.load(NEE_MODEL_PATH)
print("Loaded NEE model from:", NEE_MODEL_PATH)

nee_pred = nee_model.predict(X)
df["nee_pred"] = nee_pred.astype("float32")

print("Predicted NEE for", len(df), "rows.")

# ============================================================
# 6. Reconstruct grid from x/y and write GeoTIFF
# ============================================================

if ("x" not in df.columns) or ("y" not in df.columns):
    raise ValueError("Dataframe must contain 'x' and 'y' columns to build raster.")

xs = df["x"].to_numpy(dtype="float64")
ys = df["y"].to_numpy(dtype="float64")
vals = df["nee_pred"].to_numpy(dtype="float32")

# Unique sorted coordinates
unique_x = np.unique(xs)
unique_y = np.unique(ys)

if unique_x.size == 0 or unique_y.size == 0:
    raise ValueError("No unique x/y values found.")

# infer resolution (assuming regular grid)
if unique_x.size > 1:
    dx = float(np.median(np.diff(unique_x)))
else:
    dx = 1.0  # arbitrary for single pixel

if unique_y.size > 1:
    dy = float(np.median(np.diff(np.sort(unique_y))))
else:
    dy = 1.0

# We want row 0 at the top (max y)
ys_desc = np.sort(unique_y)[::-1]
height = len(ys_desc)
width  = len(unique_x)

x_min = unique_x.min()
y_max = ys_desc[0]

# Build transform so that pixel centers align (like rasterio.transform.xy)
transform = Affine(dx, 0, x_min - dx / 2.0,
                   0, -dy, y_max + dy / 2.0)

# Map coordinate -> index
x_to_col = {x: i for i, x in enumerate(unique_x)}
y_to_row = {y: i for i, y in enumerate(ys_desc)}

# Initialize raster with nodata as NaN
raster = np.full((height, width), np.nan, dtype="float32")

# Fill raster
for x, y, v in zip(xs, ys, vals):
    col = x_to_col.get(x)
    row = y_to_row.get(y)
    if col is None or row is None:
        continue
    raster[row, col] = v

# Prepare output directory
os.makedirs(os.path.dirname(OUT_TIF), exist_ok=True)

profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": OUT_CRS,
    "transform": transform,
    "nodata": np.nan,
    "compress": "LZW",
    "predictor": 2,
    "tiled": True,
}

with rio.open(OUT_TIF, "w", **profile) as dst:
    dst.write(raster, 1)

print("Wrote NEE prediction GeoTIFF to:", OUT_TIF)


Missing column stc_STC_0_100cm and filling with nan
Missing column LST_Day_mean and filling with nan
Missing column LST_Night_mean and filling with nan
Missing column SE_0 and filling with nan
Missing column SE_1 and filling with nan
Missing column SE_2 and filling with nan
Missing column SE_3 and filling with nan
Missing column SE_4 and filling with nan
Missing column SE_5 and filling with nan
Missing column SE_6 and filling with nan
Missing column SE_7 and filling with nan
Missing column SE_8 and filling with nan
Missing column SE_9 and filling with nan
Missing column SE_10 and filling with nan
Missing column SE_11 and filling with nan
Missing column SE_12 and filling with nan
Missing column SE_13 and filling with nan
Missing column SE_14 and filling with nan
Missing column SE_15 and filling with nan
Missing column SE_16 and filling with nan
Missing column SE_17 and filling with nan
Missing column SE_18 and filling with nan
Missing column SE_19 and filling with nan
Missing column SE_


KeyboardInterrupt



With dask

In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import math
import json

import numpy as np
import pandas as pd
import joblib
import rasterio as rio
from rasterio.transform import Affine

from dask.distributed import Client, LocalCluster
import dask.dataframe as dd

# ============================================================
# CONFIG: paths you MUST edit
# ============================================================

PARQUET_PATH = "/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined_test/predictors_2004_07.parquet"

# LOSO output paths (edit timestamp as needed)
SAVE_ROOT = "/explore/nobackup/people/spotter5/anna_v/v2/models/final_models_exp4_20251202_155619-20251202T160730Z-1-001/final_models_exp4_20251202_155619"
NEE_MODEL_PATH = os.path.join(SAVE_ROOT, "nee_model_final.joblib")
RUN_LOG_JSON   = os.path.join(SAVE_ROOT, "run_log.json")

# Output GeoTIFF
OUT_TIF = "/explore/nobackup/people/spotter5/anna_v/v2/predictions/nee_pred_2004_07.tif"

# CRS for x/y grid
OUT_CRS = "EPSG:3413"

# ============================================================
# DASK / PARALLEL CONFIG
# ============================================================

USE_DASK = True          # set False if you ever want single-core
N_WORKERS = 8            # how many workers/CPUs to use
THREADS_PER_WORKER = 1   # safer for CPU-bound LightGBM
TARGET_CHUNK_ROWS = 500_000   # approx rows per partition

# Optionally, choose a fast local scratch directory (recommended on HPC)
LOCAL_SCRATCH = "/tmp/dask-scratch"   # change if you have a better local disk

os.makedirs(LOCAL_SCRATCH, exist_ok=True)

# ============================================================
# 1. Load schema & build renaming logic
# ============================================================

# Use Dask to read parquet directly (no big in-memory pandas df yet)
ddf = dd.read_parquet(PARQUET_PATH)

# Clean column prefixes
def clean_col(c):
    return c.split("__", 1)[1] if "__" in c else c

# rename __-prefixed cols
clean_map = {c: clean_col(c) for c in ddf.columns}
ddf = ddf.rename(columns=clean_map)

# Explicit rename mapping (same as before)
rename_map = {
    'LST_Day_mean'           : 'LST_day_mean',
    'LST_Night_mean'         : 'LST_night_mean',
    'N_0_100cm'              : 'N_N_0_100cm',
    'alpha_0_100cm'          : 'alpha_ALFA_0_100cm',
    'crit_wilt_0_100cm'      : 'crit_wilt_CRIT-WILT_0_100cm',
    'field_crit_0_100cm'     : 'field_crit_FIELD-CRIT_0_100cm',
    'Fpar'                   : 'fpar',
    'ksat_0_100cm'           : 'ksat_Ksat_0_100cm',
    'Lai'                    : 'lai',
    'ormc_0_100cm'           : 'ormc_ORMC_0_100cm',
    'satfield_0_100cm'       : 'satfield_SAT-FIELD_0_100cm',
    'band1'                  : 'sm_rootzone',
    'band2'                  : 'sm_surface',
    'wcavail_0_100cm'        : 'wcavail_WCavail_0_100cm',
    'wcpf2_0_100cm'          : 'wcpf2_WCpF2_0_100cm',
    'wcpf3_0_100cm'          : 'wcpf3_WCpF3_0_100cm',
    'wcpf4_2_0_100cm'        : 'wcpf4_2_WCpF4-2_0_100cm',
    'wcres_0_100cm'          : 'wcres_WCres_0_100cm',
    'wcsat_0_100cm'          : 'wcsat_WCsat_0_100cm'
}
ddf = ddf.rename(columns=rename_map)

# ============================================================
# 2. Add month_sin / month_cos lazily in Dask
# ============================================================

if "month" not in ddf.columns:
    raise ValueError("Column 'month' not found in dataframe; cannot compute month_sin/month_cos.")

ddf = ddf.assign(
    month_sin = np.sin(2 * np.pi * ddf["month"] / 12.0),
    month_cos = np.cos(2 * np.pi * ddf["month"] / 12.0),
)

# ============================================================
# 3. Load features used in training
# ============================================================

with open(RUN_LOG_JSON, "r") as f:
    log_data = json.load(f)
FINAL_FEATURES = log_data["features_used"]

# Ensure all features exist in ddf
for col in FINAL_FEATURES:
    if col not in ddf.columns:
        ddf[col] = np.nan

# ============================================================
# 4. Load NEE model
# ============================================================

if not os.path.exists(NEE_MODEL_PATH):
    raise FileNotFoundError(f"NEE model file not found: {NEE_MODEL_PATH}")

nee_model = joblib.load(NEE_MODEL_PATH)
print("Loaded NEE model from:", NEE_MODEL_PATH)

# ============================================================
# 5. Parallel prediction with Dask (no from_pandas)
# ============================================================

def predict_partition(part: pd.DataFrame, features, model):
    """Apply model.predict to one pandas partition."""
    Xp = part[features].copy()

    for c in Xp.columns:
        if not np.issubdtype(Xp[c].dtype, np.number):
            Xp[c] = pd.to_numeric(Xp[c], errors="coerce")

    yhat = model.predict(Xp)
    part = part.copy()
    part["nee_pred"] = yhat.astype("float32")
    return part

if USE_DASK:
    print("Using Dask for parallel prediction from parquet...")

    # Compute row count once
    n_rows = int(ddf.shape[0].compute())
    if TARGET_CHUNK_ROWS is not None and TARGET_CHUNK_ROWS > 0:
        nparts = max(1, math.ceil(n_rows / TARGET_CHUNK_ROWS))
    else:
        nparts = N_WORKERS * 2

    print(f"Total rows: {n_rows:,}, target partitions: {nparts}, workers: {N_WORKERS}")

    # Repartition to desired number of partitions
    ddf = ddf.repartition(npartitions=nparts)

    cluster = LocalCluster(
        n_workers=N_WORKERS,
        threads_per_worker=THREADS_PER_WORKER,
        processes=True,
        local_directory=LOCAL_SCRATCH,
    )
    client = Client(cluster)
    print(client)

    meta = ddf._meta.assign(nee_pred=np.float32())
    ddf_pred = ddf.map_partitions(
        predict_partition,
        FINAL_FEATURES,
        nee_model,
        meta=meta
    )

    # Bring results back to a pandas DataFrame
    df_pred = ddf_pred.compute()
    client.close()
    cluster.close()
else:
    # Single-core fallback
    print("Using single-core prediction...")
    df_pred = ddf.compute()
    X_all = df_pred[FINAL_FEATURES].copy()
    for c in X_all.columns:
        if not np.issubdtype(X_all[c].dtype, np.number):
            X_all[c] = pd.to_numeric(X_all[c], errors="coerce")
    yhat_all = nee_model.predict(X_all)
    df_pred["nee_pred"] = yhat_all.astype("float32")

print("Predicted NEE for", len(df_pred), "rows.")

# ============================================================
# 6. Reconstruct grid from x/y and write LZW GeoTIFF
# ============================================================

if ("x" not in df_pred.columns) or ("y" not in df_pred.columns):
    raise ValueError("Dataframe must contain 'x' and 'y' columns to build raster.")

xs = df_pred["x"].to_numpy(dtype="float64")
ys = df_pred["y"].to_numpy(dtype="float64")
vals = df_pred["nee_pred"].to_numpy(dtype="float32")

unique_x = np.unique(xs)
unique_y = np.unique(ys)

if unique_x.size == 0 or unique_y.size == 0:
    raise ValueError("No unique x/y values found.")

if unique_x.size > 1:
    dx = float(np.median(np.diff(unique_x)))
else:
    dx = 1.0

if unique_y.size > 1:
    dy = float(np.median(np.diff(np.sort(unique_y))))
else:
    dy = 1.0

ys_desc = np.sort(unique_y)[::-1]
height = len(ys_desc)
width  = len(unique_x)

x_min = unique_x.min()
y_max = ys_desc[0]

transform = Affine(
    dx, 0, x_min - dx / 2.0,
    0, -dy, y_max + dy / 2.0
)

x_to_col = {x: i for i, x in enumerate(unique_x)}
y_to_row = {y: i for i, y in enumerate(ys_desc)}

raster = np.full((height, width), np.nan, dtype="float32")

for x, y, v in zip(xs, ys, vals):
    col = x_to_col.get(x)
    row = y_to_row.get(y)
    if col is None or row is None:
        continue
    raster[row, col] = v

os.makedirs(os.path.dirname(OUT_TIF), exist_ok=True)

profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": OUT_CRS,
    "transform": transform,
    "nodata": np.nan,
    # 🔹 LZW compression as requested
    "compress": "LZW",
    "predictor": 2,
    "tiled": True,
}

with rio.open(OUT_TIF, "w", **profile) as dst:
    dst.write(raster, 1)

print("Wrote NEE prediction GeoTIFF (LZW) to:", OUT_TIF)


Loaded NEE model from: /panfs/ccds02/nobackup/people/spotter5/anna_v/v2/models/final_models_exp4_20251202_155619-20251202T160730Z-1-001/final_models_exp4_20251202_155619/nee_model_final.joblib
Using Dask for parallel prediction from parquet...
Total rows: 25,462,012, target partitions: 51, workers: 8


/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33929 instead
  warnings.warn(


<Client: 'tcp://127.0.0.1:34773' processes=8 threads=8, memory=184.96 GiB>
Predicted NEE for 25462012 rows.
Wrote NEE prediction GeoTIFF (LZW) to: /explore/nobackup/people/spotter5/anna_v/v2/predictions/nee_pred_2004_07.tif


Make it into a function for easier reusability 

In [13]:
def predict_flux_to_geotiff(
    model_root: str,
    parquet_dir: str,
    year: int,
    month: int,
    target: str,
    out_dir: str,
    n_workers: int = 8,
    target_chunk_rows: int = 500_000,
    threads_per_worker: int = 1,
    out_crs: str = "EPSG:3413",
    local_scratch: str = "/tmp/dask-scratch",
    use_dask: bool = True,
):
    """
    Predict GPP/RECO/NEE from final LOSO LightGBM models on a parquet file and
    rasterize the output. Parquet file is automatically determined from
      parquet_dir/predictors_{year}_{month:02d}.parquet

    Parameters
    ----------
    model_root : str
        Directory containing run_log.json and *_model_final.joblib.
    parquet_path : str
        Path to predictors parquet (e.g., predictors_2004_07.parquet).
    year : int
        Year for output naming (e.g., 2004).
    month : int
        Month for output naming (1–12).
    target : {"nee", "gpp", "reco"}
        Which model to use and what to name the output.
    out_dir : str
        Directory where the GeoTIFF will be written.
        Filename will be: "{target}_pred_{year}_{month:02d}.tif".
    n_workers : int, default 8
        Number of Dask workers (processes).
    target_chunk_rows : int, default 500_000
        Approximate number of rows per Dask partition.
    threads_per_worker : int, default 1
        Threads per worker (keep 1 for CPU-bound LightGBM).
    out_crs : str, default "EPSG:3413"
        CRS for the output raster.
    local_scratch : str, default "/tmp/dask-scratch"
        Local directory for Dask temporary data (should be on local disk, not NFS).
    use_dask : bool, default True
        If False, will compute with a single core using pandas only.
    """

    # --------------------------------------------------------
    # Build parquet path automatically using year/month
    # --------------------------------------------------------
    parquet_path = os.path.join(
        parquet_dir,
        f"predictors_{year}_{month:02d}.parquet"
    )

    if not os.path.exists(parquet_path):
        raise FileNotFoundError(f"Auto-located parquet not found: {parquet_path}")

    print(f"Using parquet file: {parquet_path}")


    import math
    import json
    import numpy as np
    import pandas as pd
    import joblib
    import rasterio as rio
    from rasterio.transform import Affine

    from dask.distributed import Client, LocalCluster
    import dask.dataframe as dd

    target = target.lower()
    if target not in {"nee", "gpp", "reco"}:
        raise ValueError("target must be one of {'nee', 'gpp', 'reco'}")

    run_log_json = os.path.join(model_root, "run_log.json")
    model_path = os.path.join(model_root, f"{target}_model_final.joblib")

    os.makedirs(out_dir, exist_ok=True)
    out_tif = os.path.join(out_dir, f"{target}_pred_{year}_{month:02d}.tif")
    pred_col = f"{target}_pred"

    # ---------- Read parquet with Dask ----------
    ddf = dd.read_parquet(parquet_path)

    # ---------- Clean column names ----------
    def clean_col(c):
        return c.split("__", 1)[1] if "__" in c else c
    clean_map = {c: clean_col(c) for c in ddf.columns}
    ddf = ddf.rename(columns=clean_map)

    rename_map = {
        'LST_Day_mean'           : 'LST_day_mean',
        'LST_Night_mean'         : 'LST_night_mean',
        'N_0_100cm'              : 'N_N_0_100cm',
        'alpha_0_100cm'          : 'alpha_ALFA_0_100cm',
        'crit_wilt_0_100cm'      : 'crit_wilt_CRIT-WILT_0_100cm',
        'field_crit_0_100cm'     : 'field_crit_FIELD-CRIT_0_100cm',
        'Fpar'                   : 'fpar',
        'ksat_0_100cm'           : 'ksat_Ksat_0_100cm',
        'Lai'                    : 'lai',
        'ormc_0_100cm'           : 'ormc_ORMC_0_100cm',
        'satfield_0_100cm'       : 'satfield_SAT-FIELD_0_100cm',
        'band1'                  : 'sm_rootzone',
        'band2'                  : 'sm_surface',
        'wcavail_0_100cm'        : 'wcavail_WCavail_0_100cm',
        'wcpf2_0_100cm'          : 'wcpf2_WCpF2_0_100cm',
        'wcpf3_0_100cm'          : 'wcpf3_WCpF3_0_100cm',
        'wcpf4_2_0_100cm'        : 'wcpf4_2_WCpF4-2_0_100cm',
        'wcres_0_100cm'          : 'wcres_WCres_0_100cm',
        'wcsat_0_100cm'          : 'wcsat_WCsat_0_100cm'
    }
    ddf = ddf.rename(columns=rename_map)

    # Add month sin/cos
    ddf = ddf.assign(
        month_sin = np.sin(2 * np.pi * ddf["month"] / 12.0),
        month_cos = np.cos(2 * np.pi * ddf["month"] / 12.0),
    )

    # ---------- Load run_log.json ----------
    with open(run_log_json, "r") as f:
        log_data = json.load(f)
    final_features = log_data["features_used"]

    # Ensure all features exist
    for col in final_features:
        if col not in ddf.columns:
            ddf[col] = np.nan

    # ---------- Load model ----------
    model = joblib.load(model_path)
    print(f"Loaded {target.upper()} model: {model_path}")

    # ---------- Partition prediction ----------
    def predict_partition(part: pd.DataFrame, features, model, pred_col_name):
        Xp = part[features].copy()
        for c in Xp.columns:
            if not np.issubdtype(Xp[c].dtype, np.number):
                Xp[c] = pd.to_numeric(Xp[c], errors="coerce")
        yhat = model.predict(Xp)
        part = part.copy()
        part[pred_col_name] = yhat.astype("float32")
        return part

    # ---------- Dask parallel execution ----------
    if use_dask:
        n_rows = int(ddf.shape[0].compute())
        nparts = max(1, math.ceil(n_rows / target_chunk_rows))

        cluster = LocalCluster(
            n_workers=n_workers,
            threads_per_worker=threads_per_worker,
            processes=True,
            local_directory=local_scratch,
        )
        client = Client(cluster)
        print(client)

        ddf = ddf.repartition(npartitions=nparts)

        meta = ddf._meta.assign(**{pred_col: np.float32()})
        ddf_pred = ddf.map_partitions(
            predict_partition, final_features, model, pred_col, meta=meta
        )
        df = ddf_pred.compute()

        client.close()
        cluster.close()
    else:
        df = ddf.compute()
        X = df[final_features].copy()
        df[pred_col] = model.predict(X).astype("float32")

    print(f"Predicted {target.upper()} for {len(df):,} rows.")

    # ---------- Rasterization ----------
    xs = df["x"].to_numpy()
    ys = df["y"].to_numpy()
    vals = df[pred_col].to_numpy()

    unique_x = np.unique(xs)
    unique_y = np.unique(ys)

    dx = np.median(np.diff(unique_x)) if len(unique_x) > 1 else 1.0
    dy = np.median(np.diff(np.sort(unique_y))) if len(unique_y) > 1 else 1.0

    ys_desc = np.sort(unique_y)[::-1]
    height = len(ys_desc)
    width = len(unique_x)

    transform = Affine(
        dx, 0, unique_x.min() - dx/2,
        0, -dy, ys_desc.max() + dy/2
    )

    raster = np.full((height, width), np.nan, dtype="float32")

    x_to_col = {x: i for i, x in enumerate(unique_x)}
    y_to_row = {y: i for i, y in enumerate(ys_desc)}

    for x, y, v in zip(xs, ys, vals):
        r = y_to_row.get(y)
        c = x_to_col.get(x)
        if r is not None and c is not None:
            raster[r, c] = v

    profile = dict(
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="float32",
        crs=out_crs,
        transform=transform,
        nodata=np.nan,
        compress="LZW",
        predictor=2,
        tiled=True,
    )

    with rio.open(out_tif, "w", **profile) as dst:
        dst.write(raster, 1)

    print(f"Saved LZW-compressed raster: {out_tif}")
    return out_tif


predict_flux_to_geotiff(
    model_root="/explore/nobackup/people/spotter5/anna_v/v2/models/final_models_exp4_20251202_155619-20251202T160730Z-1-001/final_models_exp4_20251202_155619",
    parquet_dir="/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined_test",
    year=2004,
    month=7,
    target="nee",
    out_dir="/explore/nobackup/people/spotter5/anna_v/v2/predictions",
    n_workers=8,
    target_chunk_rows=500_000,
    threads_per_worker=1,
)


Using parquet file: /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined_test/predictors_2004_07.parquet
Loaded NEE model: /explore/nobackup/people/spotter5/anna_v/v2/models/final_models_exp4_20251202_155619-20251202T160730Z-1-001/final_models_exp4_20251202_155619/nee_model_final.joblib


/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42043 instead
  warnings.warn(


<Client: 'tcp://127.0.0.1:32809' processes=8 threads=8, memory=184.96 GiB>
Predicted NEE for 25,462,012 rows.
Saved LZW-compressed raster: /explore/nobackup/people/spotter5/anna_v/v2/predictions/nee_pred_2004_07.tif


'/explore/nobackup/people/spotter5/anna_v/v2/predictions/nee_pred_2004_07.tif'